### **BUILDING WHOLE GPT-2 MOE**

#### **IMPORTS NECESARIOS**

In [21]:
import os 
import sys
import torch
import urllib
import requests
import torch.nn as nn
from pathlib import Path


#### **1. INTEGRATING MOE IN TRANFORMER BLOCK**

##### **1.1 ATTENTION**

In [22]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in:int, d_out:int, context_length:int, attr:dict):

        # Initialiaze the parent
        super().__init__() 

        # Get att specific params
        num_heads:int = attr.get('num_heads', 6)
        dropout:float = attr.get('dropout', 0.1)
        qkv_bias:bool = attr.get('qkv_bias', False)

        # Get basic parameters
        self.d_out = d_out
        self.num_heads = num_heads
        # Force sum(dim(heads)) =  dout
        self.head_dim = d_out // num_heads 

        # Define the QKV matrices -> dk = dv = dout
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal = 1)   
        )

    def forward(self, x):
        # Get the shape (B, N, d_in)
        b, num_tokens, d_in = x.shape
        # Define the KQV matrices (B, num_tokens, d_out)
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We modify shapes to consider all the heads
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention 
        attn_scores = queries @ keys.transpose(2, 3)  

        # Original mask truncated to the number of tokens and converted to boolean
        # Addapted to the real amount of tokens in case input is not padded to obtain fixed N (context_length)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores -> upper diagonal MLM
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        # Get attention weights
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where d_out -> num_heads * head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

##### **1.2 NORMALIZATION**

In [23]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim:int):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x:torch.Tensor)->torch.Tensor:
        """
            Scales data per instance and makes
            it follow a ~N(0, 1)
        """
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

##### **1.3 MOE BLOCK**

In [24]:
src_path = str(Path("/home/carlo/cuarto/GPT2_MOE/src").resolve())

# Asegurar que src tiene prioridad
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

from gpt2_moe.model.moe import MOE

##### **1.4 FFN**

In [102]:
class FF(nn.Module):
    def __init__(self, d_in: int, dh: int = None):
        # Initialize parent
        super().__init__()

        # Define the parameters
        dh = dh or 4 * d_in
        self.ffn = nn.Sequential(
            nn.Linear(d_in, dh),
            nn.GELU(),
            nn.Linear(dh, d_in),
        )

    def forward(self, x:torch.Tensor)->torch.Tensor:
        
        # (B, N, d_in)->(B, N, d_in)
        output = self.ffn(x)
        print('(FFN) Computing FFN output:', output.shape)

        return output

In [ ]:
def moe_mask(layers:int, schedule:str, offset:int = 1)->list[bool]:
    """  
        Function aimed to point out 
        where MOE blocks should be 
        included. 

        The function takes the total 
        amount of layers and a MOE
        generation schedule. Then, a list 
        of bool values is returned:

        **True**: MoE in layer i

        **False**: FFN in layer i 

        ### The accepted schedules are

        **1)all**: full MoE

        **2)none**: no MoE

        **3) alternate (offset)**: set MoE with step offset

        **4)list[int]**: List of specific indices

        Params:
            layers(int):
                The amount of transformer blocks
            schedule(object):
                The moe embedding behaviour
            offset(int):
                The offset for automatic MoE embedding
                Default value is one

        Returns:
            A tuple of:
                - list[bool]: List representing MoE location
    """
    # Check if casual option matches
    if isinstance(schedule, str):
        cleaned_request = schedule.lower()

        # Check matching
        match cleaned_request:

            case 'all':
                # Return a fully true mask
                return [True]*layers
            case 'none':
                # No MoE included
                return [False]*layers
            case 'alternate':
                return [(i%offset)==0 for i in range(layers)]

    # Case manual embedding is provided -> list of indices
    # 1. Check range valid
    minimum = min(schedule)
    maximum = max(schedule)
    if (minimum>=0) and (maximum<=(layers-1)):
        return [i in schedule for i in range(layers)]
    
    print(f'Mask indices are not within layers range(0-{layers})')

In [99]:
# Define test cases
layers = 16
case_1 = 'all'
case_2 = 'none'
case_3 = 'alternate'
case_4 = [0, 1, 2, 15]
case_5 = [0,1, -1, 10]
cases = [case_1, case_2, case_3, case_4, case_5]

# Test the function 
for i, case in enumerate(cases):
    print(f'\nCASE({i})')
    res = moe_mask(layers, case, offset = 3)
    print(f'Results:', res)
    



CASE(0)
Results: [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]

CASE(1)
Results: [False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]

CASE(2)
Results: [True, False, False, True, False, False, True, False, False, True, False, False, True, False, False, True]

CASE(3)
Results: [True, True, True, False, False, False, False, False, False, False, False, False, False, False, False, True]

CASE(4)
Mask indices are not within layers range(0-16)
Results: None


##### **1.5 TRANSFORMER BLOCK**

In [100]:
class TransformerBlock(nn.Module):
    def __init__(self, attr:dict, use_moe:bool = True):
        # Initialize parent
        super().__init__()
        # Common params
        self.d_in = attr["emb_dim"]
        self.d_out = attr["emb_dim"]
        self.cl = attr["context_length"]
        # Block specific params
        self.att_related = attr["attention_related"]
        self.moe_related = attr["moe_related"]
        self.use_moe = use_moe
        # Define the attention module -> ok
        self.att = MultiHeadAttention(
           self.d_in,
           self.d_out,
           self.cl,
           self.att_related)

        # Define the ff layer
        if self.use_moe:
            self.ff = MOE(
                    self.d_in, 
                    self.d_out, 
                    self.moe_related) 
        else:
            self.ff = FF(self.d_in, 
                        attr.get('ffn_hidden'))
            

        # Define normalizations (within and previous)
        self.norm1 = LayerNorm(self.d_in)
        self.norm2 = LayerNorm(self.d_in)
        
        # Define the dropout
        self.drop_shortcut = nn.Dropout(attr['dropout'])

    def forward(self, x:torch.Tensor)->torch.Tensor:
        # Info message
        print('\nENTERING A TRANSFORMER BLOCK')
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        # (B, N, de) ->(B, N, de)
        x = self.att(x)  
        print(f'(TFBLOCK) Aspect of attention output ({x.shape})')
        x = self.drop_shortcut(x)
        # Add the original input back
        x = x + shortcut  

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        # Go through ff Layer
        x = self.ff(x)
        print(f'(TFBLOCK) Aspect of FF output ({x.shape})')
        x = self.drop_shortcut(x)
        # Add the original input back
        x = x + shortcut 

        print(f'(TFBLOCK) Aspect of block output ({x.shape})')
 

        return x

In [ ]:
# Define config for the block
block_config = {
    "context_length": 1024, 
    "emb_dim": 768, 
    "dropout":0.1, 
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    }
}

In [101]:
# Define mock tensor -> currently embedded
mock = torch.randn(size = (10, 100, 768))

# Get the block
tf_block = TransformerBlock(block_config, use_moe = False)

# Test the function
result = tf_block(mock)




ENTERING A TRANSFORMER BLOCK
(TFBLOCK) Aspect of attention output (torch.Size([10, 100, 768]))
(FFN)Computing FFN output: torch.Size([10, 100, 768])
(TFBLOCK) Aspect of FF output (torch.Size([10, 100, 768]))
(TFBLOCK) Aspect of block output (torch.Size([10, 100, 768]))


#### **2. INTEGRATING MOE IN GPT BLOCK**


##### **2.1 GPT BLOCK**

In [133]:
class GPTMOE(nn.Module):
    def __init__(self, attr:dict):
        # Initialize parents
        super().__init__()

        # Compute the moe mask
        self.moe_mask  = moe_mask(attr.get('n_layers'), **attr.get("moe_mask"))

        # Define embedding layer
        self.tok_emb = nn.Embedding(attr["vocab_size"], attr["emb_dim"])
        self.pos_emb = nn.Embedding(attr["context_length"], attr["emb_dim"])
        self.drop_emb = nn.Dropout(attr["dropout"])

        # Build the transformer blocks
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(attr, use_moe = m) for m in self.moe_mask])


        # Output layer norm
        self.final_norm = LayerNorm(attr["emb_dim"])
        # Language head
        self.out_head = nn.Linear(
            attr["emb_dim"], attr["vocab_size"], bias=False
        )

    def forward(self, in_idx:torch.Tensor)->torch.Tensor:

        # Get input dims (B, N)
        batch_size, seq_len = in_idx.shape
        # Embed the inputs (B, N, de)
        # 1. Get contextual embeddings
        tok_embeds = self.tok_emb(in_idx)
        
        # 2. Get positional embeddings
        print(f'(GPT)CREATING EMBEEDINGS')
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        print(f'(GPT)Aspect of the embeddings:{x.shape}')

        # 3. Go trought transformer blocks (B, N, de) -> (B, N, de)
        x = self.trf_blocks(x)
        print(f'(GPT)Output of the trf blocks:{x.shape}')
        x = self.final_norm(x)

        # Get logits (B, N, de) ->(B, N, |V|)
        logits = self.out_head(x)
        print(f'(GPT)Aspect of the logits space:{x.shape}')

        return logits

##### **2.2 PARAM DEFINITION**

In [120]:
# Define the parameters
model_none = {
    "vocab_size": 50257, 
    "context_length": 1024,
    "n_layers": 5,  
    "emb_dim": 768, 
    "dropout":0.1,
    "moe_mask":
        {
            "schedule": "none",
            "offset":2

        },

    "ffn_hidden_dim": None,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    },
        
    }

In [121]:
model_all = {
    "vocab_size": 50257, 
    "context_length": 1024,
    "n_layers": 5,  
    "emb_dim": 768, 
    "dropout":0.1,
    "moe_mask":
        {
            "schedule": "all",
            "offset":2

        },

    "ffn_hidden_dim": None,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    },
        
    }

In [122]:
model_interleaved = {
    "vocab_size": 50257, 
    "context_length": 1024,
    "n_layers": 10,  
    "emb_dim": 768, 
    "dropout":0.1,
    "moe_mask":
        {
            "schedule": "alternate",
            "offset":2

        },

    "ffn_hidden_dim": None,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    },
        
    }

In [124]:
model_selected = {
    "vocab_size": 50257, 
    "context_length": 1024,
    "n_layers": 8,  
    "emb_dim": 768, 
    "dropout":0.1,
    "moe_mask":
        {
            "schedule": [0,2,4,7],
            "offset":2

        },

    "ffn_hidden_dim": None,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    },
        
    }

##### **2.3 TESTING**

In [145]:
# Get the params
params_list = [model_selected, model_all, model_interleaved, model_none]

# Define mock tensor
mock = torch.randint(low = 0, high = 10, size = (10, 50))

for i, params in enumerate(params_list):
    print(f'\nTESTING NÚMERO ({i+1})')
    # Define the model 
    gpt2_moe = GPTMOE(params)
    # Call the functino
    gpt2_moe(mock)


TESTING NÚMERO (1)
(GPT)CREATING EMBEEDINGS
(GPT)Aspect of the embeddings:torch.Size([10, 50, 768])

ENTERING A TRANSFORMER BLOCK
(TFBLOCK) Aspect of attention output (torch.Size([10, 50, 768]))
(MOE) CALLING GATING MECHANISM
(router) Output of the linear:torch.Size([500, 3])
(router) Output of the top_k:torch.Size([500, 1])--(torch.Size([500, 1]))
(router) Aspect of the -inf mask:torch.Size([500, 3])
(router) BUILDING INDEXING MECHANISM
(router) ·Aspect of the first index:torch.Size([500, 1])
(router) ·Aspect of the second index:torch.Size([500, 1])
(router) ·Aspect of the later inf maskk:torch.Size([500, 3])
(router) Aspect of the normalized weights:torch.Size([500, 1])
(MOE) Aspect of the container for output tensor: torch.Size([500, 768])
(MOE) EXPERT (0)
(expert) Output of the first linear:torch.Size([151, 768])
(expert) Output of the second linear:torch.Size([151, 768])
(expert) Aspect of the expert weights: torch.Size([151, 1])
(MOE) EXPERT (1)
(expert) Output of the first line

#### **3. WEIGHT LOADING**

##### **3.1 FILE DOWNLOAD**

In [41]:
def download_gpt2(name: str, model_dir:str)->torch.Tensor:
    """  
        Function aimed to load GPT2 weights

        Params:
            name(str): 
                The name of the model
            model_dir(str):
                Base dir for model storage
            
        Returns:
            weights (torch.Tensor):
                Tensor representing weights
    
    """
    # Check path existance
    full_path = os.path.join(model_dir, name)
    if not os.path.exists(full_path):
        
        # Build the base folder 
        os.makedirs(model_dir, exist_ok = True)

        # Define the url 
        url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{name}"
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        with open(full_path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded to {full_path}")

In [32]:
# Download weights 
download_gpt2("gpt2-small-124M.pth", "weights")

##### **3.2 LOAD WEIGHTS**

In [33]:
# Get the state dict
state_dict = torch.load('weights/gpt2-small-124M.pth', map_location="cpu", weights_only=True)

In [34]:
def assign(left:torch.Tensor, right:torch.Tensor):
    """ 
        Function aimed to assign weights
    """
    if left.shape!=right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")

    # Assign weights
    return nn.Parameter(right)


In [138]:
def load_weights(model:GPTMOE, params:dict, blocks:int)->None:
    """
        Function aimed to load weights into a given 
        model. GPT skeleton architecture is expected

        Params:
            model (GPTMOE):
                The model to initialize


            params (dict):
                The dictionary containing weights

            blocks (int):
                The amount of tf blocks to initialize

        Returns:
            None:
                No value is returned

    """
    # Acquire embeddings 
    model.tok_emb.weight = assign(model.tok_emb.weight, params["tok_emb.weight"])
    model.pos_emb.weight = assign(model.pos_emb.weight, params["pos_emb.weight"])

    # Initialize tf blocks
    for i in range(blocks):
        # MHA Block
        # Initialize query
        model.trf_blocks[i].att.W_query.weight = assign(model.trf_blocks[i].att.W_query.weight,
                                                        params[f"trf_blocks.{i}.att.W_query.weight"])
        
        model.trf_blocks[i].att.W_query.bias = assign(model.trf_blocks[i].att.W_query.bias,
                                                            params[f"trf_blocks.{i}.att.W_query.bias"])
        # Initialize key 
        model.trf_blocks[i].att.W_key.weight = assign( model.trf_blocks[i].att.W_key.weight,
                                                      params[f"trf_blocks.{i}.att.W_key.weight"])
        
        model.trf_blocks[i].att.W_key.bias = assign( model.trf_blocks[i].att.W_key.bias,
                                                              params[f"trf_blocks.{i}.att.W_key.bias"])  
        # Initialize value
        model.trf_blocks[i].att.W_value.weight = assign( model.trf_blocks[i].att.W_value.weight,
                                                        params[f"trf_blocks.{i}.att.W_value.weight"]) 
        
        model.trf_blocks[i].att.W_value.bias = assign( model.trf_blocks[i].att.W_value.bias,
                                                            params[f"trf_blocks.{i}.att.W_value.bias"]) 
        # Initialize out proj
        model.trf_blocks[i].att.out_proj.weight = assign( model.trf_blocks[i].att.out_proj.weight, params[f"trf_blocks.{i}.att.out_proj.weight"])
        model.trf_blocks[i].att.out_proj.bias = assign( model.trf_blocks[i].att.out_proj.bias, params[f"trf_blocks.{i}.att.out_proj.bias"])


        # Initialize normalizations (layer norm 1 and 2)
        model.trf_blocks[i].norm1.scale = assign(model.trf_blocks[i].norm1.scale, params[f"trf_blocks.{i}.norm1.scale"])
        model.trf_blocks[i].norm1.shift = assign(model.trf_blocks[i].norm1.shift, params[f"trf_blocks.{i}.norm1.shift"])
        model.trf_blocks[i].norm2.scale = assign(model.trf_blocks[i].norm2.scale, params[f"trf_blocks.{i}.norm2.scale"])
        model.trf_blocks[i].norm2.shift = assign(model.trf_blocks[i].norm2.shift, params[f"trf_blocks.{i}.norm2.shift"])

##### **3.3 LOAD ADDITIONAL MODULES**

**Get next available name**

In [466]:
def get_next_name(path:str, key:str, ext:str = '')->str:
    """  
        Function aimed to return the next available 
        name given a base namekey. Names will be 
        built using the format name_(next_integer)

        Params:
            path(str):
                The base path to look at

            key (str):
                The base name

        Returns:
            name(str):
                The first available name
    
    """
    # Ensure path exisance
    ensure_path(path)

    # Get dir entries
    entries = [entry.name for entry in os.scandir(path)]
    last_idx = -1

    # Check entries value 
    if entries:
        # Get last valid idx
        indices = [
            int(name.split('.')[0].split('_')[-1]) for name in entries
        ]
        # Sort indices
        last_idx = sorted(indices)[-1]
        print('(next_name) Aspect of the last used idx:', last_idx)

    # Create the new idx -> if dir empty the new index will be 0
    new_idx = last_idx + 1
    print('(next_name) Aspect of the new idx:', new_idx)

    # Build the new name
    new_name = key + '_' + str(new_idx) + ext
    print('(next_name) Aspect of the new name:', new_name)
    return new_name

In [467]:
# Test the function
name_to_store = get_next_name('names_test', 'name')

(next_name) Aspect of the last used idx: 2
(next_name) Aspect of the new idx: 3
(next_name) Aspect of the new name: name_3


**Additional blocks loading**

In [330]:
def load_block(model:GPTMOE, base_path:str, block_idx:int)->None:
    """  
        Function aimed to load the a single block
        into the GPT2MOE architecture. This function won't 
        be accessed until the k experts have been individually
        trained. 

        Expected storing format for the experts and ffn the following:
        regimes/
                1/  
                    block_index/
                        expert_0.pt
                        expert_1.pt
                        |
                        |
                        expert_n.pt

                    block_index/
                        ffn.pt

        Params:
            model(GPTMOE):
                The model whose experts we want to set up
                Default value is None
            base_path (str):
                The base path where experts live
            block_idx(int):
                The block where expert is being loaded
    """

    # Put together path and block
    path = os.path.join(base_path, str(block_idx))
    print('(single_load) Aspect of the path:', path)


    # Retrieve the ff block
    block = model.trf_blocks[block_idx].ff

    # Check if we are facing moe or ff
    selected_experts = model.moe_mask[block_idx]

    # Experts selected
    if selected_experts:
        print('DEALING WITH EXPERTS')   
        # Define expert chain
        name = 'expert_{index}.pth'
        n_experts = len(block.experts)
        # Iterate over the available experts 
        for i in range(n_experts):
            print(f'(expert_load) Loading expert ({i})')
            # Get to the expert i 
            full_path = os.path.join(path, name.format(index = i))
            print(' ·Aspect of the full path:', full_path)
            # Retrieve the state dict of the expert
            expert_state_dict = torch.load(full_path, weights_only = True)
            # Load the state dict into a selected model's module
            block.experts[i].load_state_dict(expert_state_dict)

    
    # FFN selected
    else:
        print('DEALING WITH FFN')
        # Define expert chain
        name = 'ffn.pth'
        # Get the full path to the ffn
        full_path = os.path.join(path, name)
        print(' ·(ffn)Aspect of the full path:', full_path)
        # Retrieve the state dict of the ffn
        ffn_state_dict = torch.load(full_path, weights_only = True)
        # Load the state dict into a selected model's module
        block.ffn.load_state_dict(ffn_state_dict)
         

In [ ]:
def load_all(model:GPTMOE, base_path:str)->None:
    """ 
        Function aimed to load all the additional blocks
        added to the base GPT2 skeleton. This function won't 
        be accessed until the k experts have been individually
        trained. 

        Expected storing format for the experts and ffn is the following:
        regimes/
                1/  
                    block_index/
                        expert_0.pt
                        expert_1.pt
                        |
                        |
                        expert_n.pt

                    block_index/
                        ffn.pt

        Params:
            model(GPTMOE):
                The model whose experts we want to set up
            base_path (str):
                The base path where experts live
    
    """

    # Iterate over all blocks
    for i in range(len(model.trf_blocks)):
        # Loading each block
        print(f'(blocks_loader)LOADING BLOCK ({i})')
        load_block(model, base_path = base_path, block_idx = i)

In [332]:
# Define moe to have experts at [0] and ffn at [1]
model_selected = {
    "vocab_size": 50257, 
    "context_length": 1024,
    "n_layers": 2,  
    "emb_dim": 768, 
    "dropout":0.1,
    "moe_mask":
        {
            "schedule": [0],
            "offset":2

        },

    "ffn_hidden_dim": None,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":True
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    },
        
    }

In [333]:
# Create the model to modify
model = GPTMOE(model_selected)

In [334]:
# Define mock blocks in storage (ones, ones)
mock = torch.ones(768, 768)
model.trf_blocks[0].ff.experts[0].l1.weight = nn.Parameter(mock)
mock = torch.ones(768, 3072)
model.trf_blocks[1].ff.ffn[2].weight = nn.Parameter(mock)

# Create mock expert's state dicts
for i in range(0,3):
    torch.save(model.trf_blocks[0].ff.experts[0].state_dict(),f'regimes/1/0/expert_{i}.pth')
# Create mock ffn state dict
torch.save(model.trf_blocks[1].ff.ffn.state_dict(), f'regimes/1/1/ffn.pth')

In [335]:
# Create the model again to clean previous mocking
model = GPTMOE(model_selected)

# Load the weights
load_all(model, base_path = "regimes/1/")


(blocks_loader)LOADING BLOCK (0)
(single_load) Aspect of the path: regimes/1/0
DEALING WITH EXPERTS
(expert_load) Loading expert (0)
 ·Aspect of the full path: regimes/1/0/expert_0.pth
(expert_load) Loading expert (1)
 ·Aspect of the full path: regimes/1/0/expert_1.pth
(expert_load) Loading expert (2)
 ·Aspect of the full path: regimes/1/0/expert_2.pth
(blocks_loader)LOADING BLOCK (1)
(single_load) Aspect of the path: regimes/1/1
DEALING WITH FFN
 ·(ffn)Aspect of the full path: regimes/1/1/ffn.pth


In [336]:
# Check loading -> experts
print(model.trf_blocks[0].ff.experts[0].l1.weight)
print(model.trf_blocks[0].ff.experts[1].l1.weight)
print(model.trf_blocks[0].ff.experts[2].l1.weight)

# Check loading -> ffn
print(model.trf_blocks[1].ff.ffn[2])




Parameter containing:
tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]], requires_grad=True)
Parameter containing:
tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]], requires_grad=True)
Parameter containing:
tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]], requires_grad=True)
Linear(in_features=3072, out_features=768, bias=True)


#### **4. PARAMS FREEZING**

In [ ]:
def freeze_selective(model:GPTMOE, train_mode:bool)->None:
    """
        Function aimed to selectively freeze
        model parameters.

        The function has the following behaviours:

        1) freeze base: experts training and don't mind router
        2) freeze base and experts: router training
    
        Params:
            model(GPTMOE):
                The model to work with
            train_mode (bool):
                Indicator for expert or router training
        Returns:
            None:
                No value is returned

    """

    # Freeze the base model 
    model.requires_grad_(False)

    # MOE-FFN training --> unfreeze them 
    if train_mode:
        # Define trainable blocks list 
        trainable = [block.ff.requires_grad_(True) for block in model.trf_blocks]

    else:
        # Training the router
        trainable = [block.ff.gate.requires_grad_(True) for block in model.trf_blocks if hasattr(block.ff, 'gate')]

In [398]:
# Freeze model  unfreeze experts
# Load model
model = GPTMOE(model_selected)
# Call function over model 
freeze_selective(model, train_mode = True)
# Check it
counter = 0
counter_all = 0
for i, param in enumerate(model.named_parameters()):
    if 'ff' in param[0]:
        print(param[0],':', param[1].requires_grad)
        counter+=1

    if param[1].requires_grad:
        counter_all+=1
print('Totales:', counter_all)
print('Esperados:', counter)


trf_blocks.0.ff.gate.linear.weight : True
trf_blocks.0.ff.gate.linear.bias : True
trf_blocks.0.ff.experts.0.l1.weight : True
trf_blocks.0.ff.experts.0.l1.bias : True
trf_blocks.0.ff.experts.0.l2.weight : True
trf_blocks.0.ff.experts.0.l2.bias : True
trf_blocks.0.ff.experts.1.l1.weight : True
trf_blocks.0.ff.experts.1.l1.bias : True
trf_blocks.0.ff.experts.1.l2.weight : True
trf_blocks.0.ff.experts.1.l2.bias : True
trf_blocks.0.ff.experts.2.l1.weight : True
trf_blocks.0.ff.experts.2.l1.bias : True
trf_blocks.0.ff.experts.2.l2.weight : True
trf_blocks.0.ff.experts.2.l2.bias : True
trf_blocks.1.ff.ffn.0.weight : True
trf_blocks.1.ff.ffn.0.bias : True
trf_blocks.1.ff.ffn.2.weight : True
trf_blocks.1.ff.ffn.2.bias : True
Totales: 18
Esperados: 18


In [401]:
# Freeze model and experts unfreeze router
# Load model
model = GPTMOE(model_selected)
# Call function over model 
freeze_selective(model, train_mode = False)
# Check it
counter = 0
counter_all = 0
for i, param in enumerate(model.named_parameters()):
    if 'ff.gate.linear' in param[0]:
        print(param[0],':', param[1].requires_grad)
        counter+=1
    if param[1].requires_grad:
        counter_all+=1
print('Totales:', counter_all)
print('Esperados:', counter)


trf_blocks.0.ff.gate.linear.weight : True
trf_blocks.0.ff.gate.linear.bias : True
Totales: 2
Esperados: 2


#### **5. PARAMS STORAGE**

In [460]:
def ensure_path(path:str)->None:
    """  
        Function aimed to ensure path existance
        before storing

        Params:
            path(str):
                The path to check
        Returns:
            None:
                No value is returned
    
    """

    # Ensure path existance
    os.makedirs(path, exist_ok = True)

In [459]:
def params_storage(model:GPTMOE, base_path:str, train_mode:bool)->None:
    """
        Function aimed to store data based on 
        the training mode that is being conducted

        Train modes are the following:

        1) experts(True): only experts are stored
        2) routers(False): the whole model is stored

        Params:
            model(GPTMOE):
                The model we are currently using
            base_path(str):
                The base path to store data at
            train_mode(bool):
                The train mode we are following

    """

    # Switch train mode 
    if train_mode:
        # Save the experts and ffn only
        for i, block in enumerate(model.trf_blocks):
            # Define the full_path
            full_path = os.path.join(base_path, str(i))
            # Check mask 
            if model.moe_mask[i]:
                # We are training one expert at a time
                name = get_next_name(full_path, 'expert', ext = '.pth')
                state_dict = block.ff.experts[i].state_dict()
            else:
                # Define ffn name and state dict
                name = get_next_name(full_path, 'ffn', ext = '.pth')
                state_dict = block.ff.ffn.state_dict()
      
            # Store transparently
            torch.save(state_dict, os.path.join(full_path, name))
    else:
        # Define path
        path = os.path.join(base_path, 'model')
        # Get model's available name
        name = get_next_name(path, 'model', ext = '.pth')
        path_name = os.path.join(path, name)
        # Store the whole model with the trained router
        torch.save(name, path_name)

In [470]:
# Check how it works -> model of two blocks (moe, ffn)
model = GPTMOE(model_selected)
# Define the base path
base_path = 'regimes_test/1'

# Store the experts and ffn 
train_mode = True
params_storage(model = model, base_path = base_path, train_mode = train_mode)




(next_name) Aspect of the last used idx: 0
(next_name) Aspect of the new idx: 1
(next_name) Aspect of the new name: expert_1.pth
(next_name) Aspect of the last used idx: 0
(next_name) Aspect of the new idx: 1
(next_name) Aspect of the new name: ffn_1.pth


In [472]:
# Check how it works -> model of two blocks (moe, ffn)
model = GPTMOE(model_selected)
# Define the base path
base_path = 'regimes_test/1'

# Store the experts and ffn 
train_mode = False
params_storage(model = model, base_path = base_path, train_mode = train_mode)



(next_name) Aspect of the last used idx: 0
(next_name) Aspect of the new idx: 1
(next_name) Aspect of the new name: model_1.pth
